In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Reading From Bronze

###Table - cust_info

In [0]:
df = spark.table("lakehouse.bronze.cust_info")

## Data Transformation

In [0]:
# trim the stings (columns)
# normalization for maritial status and gender
# names also need to trim
df.display()

### Trimming The Columns 

In [0]:
str_columns = ['cst_id','cst_key','cst_firstname','cst_lastname',
               'cst_marital_status','cst_gndr','cst_create_date']
for c in str_columns:
    df = df.withColumn(c, trim(col(c)))

display(df.head(5))

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06


### **Cleaning The Whole Table Using Spark.Sql By Writing Sql Query**

In [0]:
query = """
select
  cst_id as cust_id,
  cst_key as cust_key,
  trim(cst_firstname) as first_name,
  trim(cst_lastname) as last_name,
  (case when cst_marital_status== 'M' then 'Married' else 'Single' end) as marital_status,
  (case when cst_gndr == 'M' then 'Male' else 'Female' end) as gender,
  cst_create_date as create_date
from lakehouse.bronze.cust_info
"""

df = spark.sql(query)

### Renaming All Columns Using Mappinng Otherr Than Sql

In [0]:
mapping = {
  'cst_id' : 'cust_id',
  'cst_key' : 'cust_key',
  'cst_firstname' : 'first_name',
  'cst_lastname' : 'last_name',
  'cst_marital_status' : 'marital_status',
  'cst_gndr' : 'gender',
  'cst_create_date' : 'create_date'
}
df = df.select([
    df[c].alias(mapping.get(c, c))
    for c in df.columns
])
display(df.head(2))

cust_id,cust_key,first_name,last_name,marital_status,gender,create_date
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06


### NUlls 

In [0]:
df.select([

sum(col(c).isNull().cast('int')).alias(c)
for c in df.columns    
]).display()

cust_id,cust_key,first_name,last_name,marital_status,gender,create_date
4,0,8,7,0,0,4


### i filtered out all the not nulls rows then overwrite it in df

In [0]:
df = df.filter("cust_id IS NOT NULL")
df.filter("cust_id IS NULL").show()

+-------+--------+----------+---------+--------------+------+-----------+
|cust_id|cust_key|first_name|last_name|marital_status|gender|create_date|
+-------+--------+----------+---------+--------------+------+-----------+
+-------+--------+----------+---------+--------------+------+-----------+



In [0]:
df = df.fillna({'first_name':'Unknown'})
df.filter('first_name is null').show()
df = df.fillna({'last_name':'Unknown'})
df.filter('last_name is null').show()

+-------+--------+----------+---------+--------------+------+-----------+
|cust_id|cust_key|first_name|last_name|marital_status|gender|create_date|
+-------+--------+----------+---------+--------------+------+-----------+
+-------+--------+----------+---------+--------------+------+-----------+

+-------+--------+----------+---------+--------------+------+-----------+
|cust_id|cust_key|first_name|last_name|marital_status|gender|create_date|
+-------+--------+----------+---------+--------------+------+-----------+
+-------+--------+----------+---------+--------------+------+-----------+



In [0]:
# df.select("cust_id").distinct().count()
# df.select('cust_id').dropDuplicates().count()
# # dim_cust.display()

df.groupBy("cust_id") \
           .count() \
           .filter("count > 1") \
           .show()

+-------+-----+
|cust_id|count|
+-------+-----+
|  29483|    2|
|  29473|    2|
|  29449|    2|
|  29466|    3|
|  29433|    2|
+-------+-----+



### Normalising marital_status and gender using pyspark

In [0]:
df = df.withColumn("marital_status",
              when(col('marital_status') == 'M', 'Married').otherwise('Single')\
    ).withColumn("gender", 
              when(col("gender") == "M", "Male").otherwise('Female'))
    
display(df.head(2))

cust_id,cust_key,first_name,last_name,marital_status,gender,create_date
11000,AW00011000,Jon,Yang,Single,Female,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Female,2025-10-06


## Write Into Silver Table

In [0]:
df.write.mode("overwrite")\
.format('delta')\
.option('overwriteSchema', 'True')\
.saveAsTable('lakehouse.silver.s_cust_info')

In [0]:
%sql
DESCRIBE DETAIL lakehouse.bronze.cust_info

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics
delta,4917a6bd-a253-411a-b1f0-eb99b6c81238,lakehouse.bronze.cust_info,null,abfss://datalake@romaandatalake.dfs.core.windows.net/Lakehouse/__unitystorage/catalogs/08b6f89d-ca00-4592-8f67-fb33ffc75f9f/tables/27ab4308-669a-4402-bc47-53be14d0d864,2026-06-03T18:42:21.744Z,2026-06-06T09:13:57Z,List(),List(),1,108225,Map(delta.enableDeletionVectors -> true),3,7,List(deletionVectors),"Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)"


### Table - prd_info_1

In [0]:
prd_info_1  =  spark.table("lakehouse.bronze.prd_info_1")


In [0]:
# found nulls in some columns 
# inconsistent date format 
prd_info_1.display()

In [0]:
prd_info_1.printSchema()

root
 |-- prd_id: integer (nullable = true)
 |-- prd_key: string (nullable = true)
 |-- prd_name: string (nullable = true)
 |-- prd_cost: integer (nullable = true)
 |-- prd_line: string (nullable = true)
 |-- prd_start_date: date (nullable = true)



In [0]:
prd_info_1.describe().display()

summary,prd_id,prd_key,prd_name,prd_cost,prd_line
count,295,295,295,295,295
mean,440.3254237288136,null,null,432.43728813559323,null
stddev,108.97438941657072,null,null,532.7724514268953,null
min,210,BB-7421,AWC Logo Cap,1,M
max,606,WB-H098,Women's Tights- S,2171,T


## finding nulls 


In [0]:
prd_info_1.select([

sum(col(c).isNull().cast('int')).alias(c)    
for c in prd_info_1.columns
]).display()

prd_id,prd_key,prd_name,prd_cost,prd_line,prd_start_date
0,0,0,0,0,0


# filled null with meadian in prd_cost

In [0]:
prd_info_1 = prd_info_1.fillna({"prd_cost" : 202})

In [0]:
display(prd_info_1.head(5))

prd_id,prd_key,prd_name,prd_cost,prd_line,prd_start_date
408,HB-R956,HL Road Handlebars,53,R,2013-07-01
486,ST-1401,All-Purpose Bike Stand,59,M,2013-07-01
504,FR-T67U-58,LL Touring Frame - Blue- 58,200,T,2013-07-01
503,FR-T67U-54,LL Touring Frame - Blue- 54,200,T,2013-07-01
502,FR-T67U-50,LL Touring Frame - Blue- 50,200,T,2013-07-01


### replacing the column names using spark sql 

In [0]:
prd_info_1.columns

['prd_id', 'prd_key', 'prd_name', 'prd_cost', 'prd_line', 'prd_start_date']

In [0]:
prd_info_1.createOrReplaceTempView("prd_info_1")

prd_info_1 = spark.sql("""select
                        prd_id as prd_id,
                        prd_key,
                        prd_name,
                        prd_cost as prd_cost,
                        trim(prd_line) as prd_line
                    from prd_info_1""")

### No Duplicate Rows Found

In [0]:
prd_info_1.groupBy(prd_info_1.columns).count().filter("count > 1").display()
display(prd_info_1.distinct().count())

prd_id,prd_key,prd_name,prd_cost,prd_line,count


295

In [0]:
%sql
select 
mode(prd_line) as prd_line_mode
from prd_info_1;


prd_line_mode
R


In [0]:
prd_info_1 = prd_info_1.fillna({'prd_line': 'R'})

In [0]:
# prd_info_1.filter(col("prd_end_date") < col("prd_start_date")).display()


In [0]:
# %sql
# SELECT
#     prd_id,
#     prd_key,
#     prd_nm AS prd_name,
#     prd_cost AS prd_cost,
#     trim(prd_line) AS prd_line,

#     CASE
#         WHEN prd_start_dt > prd_end_dt
#         THEN prd_end_dt
#         ELSE prd_start_dt
#     END AS prd_start_date,

#     CASE
#         WHEN prd_start_dt > prd_end_dt
#         THEN prd_start_dt
#         ELSE prd_end_dt
#     END AS prd_end_date

# FROM lakehouse.bronze.prd_info_1
# limit 5

In [0]:
prd_info_1.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lakehouse.silver.s_prd_info_1")

### Table - 3 -sales_details


In [0]:
sales_details = spark.table('lakehouse.bronze.sales_details')

In [0]:
display(sales_details.head(5))

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
SO43697,BK-R93R-62,21768,20101229,20110105,20110110,3578,1,3578
SO43698,BK-M82S-44,28389,20101229,20110105,20110110,3400,1,3400
SO43699,BK-M82S-44,25863,20101229,20110105,20110110,3400,1,3400
SO43700,BK-R50B-62,14501,20101229,20110105,20110110,699,1,699
SO43701,BK-M82S-44,11003,20101229,20110105,20110110,3400,1,3400


In [0]:
sales_details.select([

sum(col(c).isNull().cast("int")).alias(c)
for c in sales_details.columns    
]).display()

sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
0,0,0,0,0,0,8,0,7


In [0]:
# found nulls in price
# found negatice sales in sls_sales,
# found negative price in sls_price
sales_details.describe().display()

summary,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
count,60398,60398,60398,60398,60398,60398,60390,60398,60391
mean,null,null,18841.685420047022,2.0123402797178715E7,2.0129938434186563E7,2.013010416608166E7,486.0987249544627,1.0004139209907612,486.0081303505489
stddev,null,null,5432.430404014172,356972.2830439616,4851.0607082607285,4985.453229346541,928.5008670773042,0.04401148687509096,928.5383799019598
min,SO43697,BC-M005,11000,0,20110105,20110110,-54,1,-1701
max,SO75123,WB-H098,29483,20140128,20140204,20140209,3578,10,3578


### sls_sales 

In [0]:
sales_details.filter(col('sls_sales') > 0).select(median("sls_sales")).show()

+-----------------+
|median(sls_sales)|
+-----------------+
|             30.0|
+-----------------+



In [0]:
sales_details = sales_details.withColumn("sls_sales", when(col('sls_sales') <= 0, 30).otherwise(col('sls_sales')))

In [0]:
sales_details = sales_details.fillna(30)
sales_details.filter('sls_sales is null').show()

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+



### sls_price

In [0]:
sales_details = sales_details.withColumn('sls_price', col("sls_sales") * col("sls_quantity"))

In [0]:
sales_details.show()

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|    20101229|   20110105|  20110110|     3578|           1|     3578|
|    SO43698| BK-M82S-44|      28389|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43699| BK-M82S-44|      25863|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43700| BK-R50B-62|      14501|    20101229|   20110105|  20110110|      699|           1|      699|
|    SO43701| BK-M82S-44|      11003|    20101229|   20110105|  20110110|     3400|           1|     3400|
|    SO43702| BK-R93R-44|      27645|    20101230|   20110106|  20110111|     3578|           1|     3578|
|    SO43703| BK-R93R-62|      16624|

In [0]:
from pyspark.sql.functions import to_date

sales_details = sales_details.withColumn('sls_order_dt', to_date('sls_order_dt', 'yyyyMMdd'))\
    .withColumn('sls_ship_dt', to_date('sls_ship_dt', 'yyyyMMdd'))\
    .withColumn('sls_due_dt', to_date('sls_due_dt', 'yyyyMMdd'))
sales_details.show()

+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|sls_ord_num|sls_prd_key|sls_cust_id|sls_order_dt|sls_ship_dt|sls_due_dt|sls_sales|sls_quantity|sls_price|
+-----------+-----------+-----------+------------+-----------+----------+---------+------------+---------+
|    SO43697| BK-R93R-62|      21768|  2010-12-29| 2011-01-05|2011-01-10|     3578|           1|     3578|
|    SO43698| BK-M82S-44|      28389|  2010-12-29| 2011-01-05|2011-01-10|     3400|           1|     3400|
|    SO43699| BK-M82S-44|      25863|  2010-12-29| 2011-01-05|2011-01-10|     3400|           1|     3400|
|    SO43700| BK-R50B-62|      14501|  2010-12-29| 2011-01-05|2011-01-10|      699|           1|      699|
|    SO43701| BK-M82S-44|      11003|  2010-12-29| 2011-01-05|2011-01-10|     3400|           1|     3400|
|    SO43702| BK-R93R-44|      27645|  2010-12-30| 2011-01-06|2011-01-11|     3578|           1|     3578|
|    SO43703| BK-R93R-62|      16624|

In [0]:
sales_details.write.mode("overwrite")\
.format('delta')\
.option('overwriteSchema', 'True')\
.saveAsTable('lakehouse.silver.s_sales_details')

# Completed the Silver Layer